# Day 21/42: Gradient Boosting with XGBoost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day21_gradient_boosting_xgboost/day21_notebook.ipynb)

## What You'll Learn

- The real difference between bagging (Random Forest) and boosting (XGBoost)
- How to watch a boosted model fit the previous trees' mistakes, round by round
- Why `learning_rate` is the hyperparameter that matters most
- How to explain individual predictions with SHAP, not just a feature importance bar chart

## Dataset used

A synthetic classification dataset built in this notebook with `make_classification`, no download needed.

Run every cell top to bottom. The first code cell installs `xgboost` and `shap` if they're not already available, no manual setup needed in Colab or a fresh environment.

## Setup

In [ ]:
# Installs xgboost and shap if not already present. Safe to run every time.
%pip install -q xgboost shap

In [ ]:
%matplotlib inline
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from xgboost import XGBClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
print("Setup complete. xgboost", xgb.__version__, "| shap", shap.__version__)

## The Concept

**Bagging** (Random Forest, Day 17): build many trees independently, in parallel, each on a random sample of rows and features. Average their votes. Each tree never sees what the others got wrong.

**Boosting** (XGBoost): build trees one at a time, in sequence. Each new tree is trained specifically to correct the errors the previous trees made. Tree 50 exists because trees 1 through 49 weren't perfect. That sequential correction is the entire idea.

This isn't a minor implementation detail. It changes how the model behaves: boosting usually reaches higher accuracy on structured, tabular data, but it can overfit if you let it run too many rounds, since each new tree is chasing the remaining errors ever more closely, including the noise.

XGBoost specifically adds regularization and a more efficient tree-building algorithm on top of plain gradient boosting, which is part of why 17 of the 29 Kaggle competition winning solutions published in 2015 used XGBoost, with deep neural networks the second most common approach at 11 solutions, according to the original XGBoost paper by Chen and Guestrin.

## 1. Bagging vs Boosting, Same Dataset

Train a Random Forest and an XGBoost model on identical data and compare accuracy, training time, and cross-validated stability (tying back to Day 20).

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10, n_redundant=5,
    n_classes=2, flip_y=0.05, class_sep=0.8, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_time = time.time() - start
rf_acc = accuracy_score(y_test, rf.predict(X_test))

start = time.time()
xgb_clf = XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")
xgb_clf.fit(X_train, y_train)
xgb_time = time.time() - start
xgb_acc = accuracy_score(y_test, xgb_clf.predict(X_test))

print(f"Random Forest (bagging): accuracy={rf_acc:.3f}, train time={rf_time:.2f}s")
print(f"XGBoost (boosting):      accuracy={xgb_acc:.3f}, train time={xgb_time:.2f}s")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X, y, cv=skf, scoring="accuracy")
xgb_cv = cross_val_score(XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"), X, y, cv=skf, scoring="accuracy")

print(f"Random Forest CV: {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}")
print(f"XGBoost CV:       {xgb_cv.mean():.3f} +/- {xgb_cv.std():.3f}")

On this dataset the two land close on raw accuracy, but notice the CV standard deviation: XGBoost's score moves around less across folds. That's a common pattern, not a guarantee. The right answer to "which one is better" is almost always "tune both and compare," not a rule you can apply blind.

## 2. Watching Boosting Actually Boost

This is the part a single accuracy number can't show you: each new tree exists to fix what came before. Track the loss after every single tree is added, on both the training set and a held-out test set.

In [ ]:
xgb_eval = XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    random_state=42, eval_metric="logloss"
)
xgb_eval.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

results = xgb_eval.evals_result()
train_loss = results["validation_0"]["logloss"]
test_loss = results["validation_1"]["logloss"]

print("Loss after tree 1:   train =", round(train_loss[0], 4), " test =", round(test_loss[0], 4))
print("Loss after tree 50:  train =", round(train_loss[49], 4), " test =", round(test_loss[49], 4))
print("Loss after tree 100: train =", round(train_loss[-1], 4), " test =", round(test_loss[-1], 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_loss, label="Train loss", color="#7C4DFF")
plt.plot(test_loss, label="Test loss", color="#00C4CC")
plt.xlabel("Boosting round (tree number)")
plt.ylabel("Log loss")
plt.title("Each Tree Fits the Previous Trees' Mistakes")
plt.legend()
plt.tight_layout()
plt.show()

Train loss keeps dropping the entire time, tree 100 is still finding new mistakes to correct on data it has memorized. Test loss drops fast early, then flattens, then creeps back up slightly past round 60 or so. That uptick is overfitting, the exact pattern from Day 18, just visible here one tree at a time instead of one model complexity setting at a time. This is why `early_stopping_rounds` exists: stop adding trees once test loss stops improving.

## 3. The Hyperparameter That Matters Most: learning_rate

`learning_rate` controls how much each new tree is allowed to correct. Low values mean each tree makes a small, cautious adjustment, more trees are needed to reach the same fit. High values mean each tree corrects aggressively, which can overshoot and add noise faster.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.3, 0.5]
lr_results = []

for lr in learning_rates:
    clf = XGBClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42, eval_metric="logloss")
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    lr_results.append({"learning_rate": lr, "test_accuracy": round(acc, 3)})

pd.DataFrame(lr_results)

At `n_estimators=100` fixed, the lowest learning rate (0.01) underperforms; 100 small, cautious corrections simply haven't gotten the model far enough yet. The fix isn't necessarily a higher learning rate, it's often a lower learning rate paired with more trees and `early_stopping_rounds`, which tends to generalize better than a high learning rate racing to a worse local optimum. Learning rate and number of trees are a pair, never tune one without considering the other.

## 4. Explaining Predictions: SHAP

`feature_importances_` tells you which features mattered on average, across the whole dataset. It can't tell you why one specific prediction came out the way it did, or whether a feature pushed a prediction up or down. SHAP (SHapley Additive exPlanations) answers both.

In [ ]:
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

final_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, eval_metric="logloss")
final_model.fit(X_train_df, y_train)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test_df[:200])  # subset for speed

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[::-1][:5]

print("Top 5 features by mean |SHAP value|:")
for idx in top5_idx:
    print(f"  {feature_names[idx]}: {mean_abs_shap[idx]:.4f}")

In [ ]:
shap.summary_plot(shap_values, X_test_df[:200], plot_size=(8, 5))

Each dot is one prediction. Color shows whether that feature's value was high (pink) or low (blue) for that row. Position left or right shows whether it pushed the prediction down or up. This is the level of detail you need when a stakeholder, or a regulator, asks "why did the model flag this specific case," not just "what mattered overall." A bar chart of feature importances can't answer that question. SHAP can.

## 5. Practice: Bagging or Boosting?

Four scenarios below. For each one, decide whether you'd reach for a bagged model (Random Forest) or a boosted one (XGBoost) as your first attempt, and why.

In [ ]:
scenarios = [
    {"name": "Scenario A", "desc": "Tabular loan default data, 50 features, need a fast baseline today, some missing values."},
    {"name": "Scenario B", "desc": "Noisy sensor data, you suspect a few mislabeled rows, want a model resistant to outlier rows."},
    {"name": "Scenario C", "desc": "Kaggle-style competition, leaderboard score is everything, you have time to tune carefully."},
    {"name": "Scenario D", "desc": "You need every prediction explained to a regulator, simplicity matters more than 1% accuracy."},
]

for s in scenarios:
    print(f"{s['name']}: {s['desc']}")

**Your turn.** Write Bagging or Boosting for each scenario, with one line of reasoning, before checking below.

- Scenario A: 
- Scenario B: 
- Scenario C: 
- Scenario D: 

### Solution

**Scenario A: Either, lean XGBoost.** XGBoost natively handles missing values without a separate imputation step and trains fast. A solid first baseline.

**Scenario B: Random Forest.** Bagging averages over many independent trees, which dilutes the influence of a handful of mislabeled rows. Boosting can chase those same mislabeled rows hard, treating them as "mistakes to correct" round after round.

**Scenario C: XGBoost (or another boosting variant like LightGBM/CatBoost).** This is exactly the setting that produced the 17-of-29 Kaggle statistic above: time to tune carefully, evaluation is purely about leaderboard accuracy, and boosting tends to extract the last percentage points that bagging leaves on the table.

**Scenario D: Random Forest, or a much shallower XGBoost.** Both are technically "black box" without SHAP, but a Random Forest's individual trees are easier to reason about directly, and its behavior is generally more stable and predictable to explain in plain terms to a non-technical reviewer.

## 6. Try It Yourself

No solution provided here. Make these changes and see what happens:

1. In Section 2, add `early_stopping_rounds=10` to the `.fit()` call (you'll need an `eval_set`). At which tree number does training actually stop?
2. In Section 3, fix `learning_rate=0.01` and increase `n_estimators` to 500. Does test accuracy catch up to the higher learning rates?
3. In Section 4, pick a different feature near the bottom of the SHAP summary plot and explain in one sentence why it might matter less than `feature_4`.

## Self-Check Before Day 22

You're ready to move on if you can answer these without scrolling back up:

1. In one sentence, what does tree number 50 in a boosted model actually exist to do?
2. Why does Random Forest tend to handle a few mislabeled training rows better than XGBoost does, by default?
3. What does the test loss curve in Section 2 tell you that a single final accuracy number wouldn't?
4. Name one thing SHAP can tell you about a prediction that `feature_importances_` cannot.
5. Why are `learning_rate` and `n_estimators` almost always tuned together rather than separately?

If any of these feel shaky, re-run the relevant section above before starting Day 22.

## What's Next

Tomorrow, Day 22: K-Means Clustering. Everything so far has been supervised, a labeled target to predict. K-Means flips that: no labels at all, just rows of data and the question of which ones naturally group together.

Full series repo: https://github.com/VaishnaviJagtap18/-42-Days-of-ML-Challenge

#42DaysOfML #MachineLearning #MLEngineer #Python #DataScience